# Notebook 03 - Optuna Hyperparameter Tuning for GRU and TCN

Perform hyperparameter tuning on the two best models from Notebook 02

Strategy:

1. **Coarse Search:** wide search on 3 LOSO-Folds.
2. **Refined Search:** narrow search around the most promising coarse parameters using 5 LOSO-Folds.
3. **Final Check:** Compare baseline- and tuned-configuration on all 10 LOSO-Folds

## 1. Setup

In [ ]:
# Core imports
import os
import re
import gc
import json
import time
import random
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TensorFlow / Keras
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Disable XLA/JIT to avoid long graph compilation during repeated fold training.
tf.config.optimizer.set_jit(False)

# Let TensorFlow allocate GPU memory gradually.
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("XLA/JIT enabled:", tf.config.optimizer.get_jit())

## 2. Configuration

In [ ]:
# Experiment setup
WINDOW_SIZE = 30
STEP_SIZE = 10
TARGET_COLUMN = "Energy expenditures (W/kg)"
TIME_COLUMN = "Time (s)"
ACTIVITY_COLUMN = "Activity Code"
SUBJECT_COLUMN = "Subject"
WKG_TO_KCAL_PER_MIN = 60 / 4184

SUBJECT_WEIGHTS_KG = {
    1: 63.49,
    2: 63.49,
    3: 71.20,
    4: 68.03,
    5: 68.03,
    6: 68.03,
    7: 95.24,
    8: 65.76,
    9: 68.93,
    10: 58.05,
}

INPUT_SIGNALS = [
    "Waist Acceleration",
    "Chest Acceleration",
    "Left Ankle Acceleration",
    "right Ankle Acceleration",
    "left wrist Acceleration",
    "right wrist Acceleration",
    "EMG_magnitude_left",
    "EMG_magnitude_right",
    "left wrist electrodermal",
    "right wrist electrodermal",
    "left wrist Temperature",
    "right wrist Temperature",
    "Breath Frequency",
    "Minute Ventilation",
    "SpO2",
    "Heart Rate",
]

# Change this only if Kaggle cannot find the data automatically.
MANUAL_DATA_DIR = None

OUTPUT_DIR = Path("/kaggle/working/notebook_03c_optuna_tuning")
if not Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("notebook_03c_optuna_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Number of input signals:", len(INPUT_SIGNALS))
print("Output directory:", OUTPUT_DIR)

## 3. Load and Prepare Data

In [ ]:
def parse_subject_id(path):
    """Extract the subject id from filenames such as Subject (3).csv."""
    match = re.search(r"Subject \((\d+)\)", path.name)
    if match is None:
        raise ValueError(f"Could not parse subject id from {path.name}")
    return int(match.group(1))


def find_data_dir():
    """Find the folder containing the subject CSV files."""
    candidates = []
    if MANUAL_DATA_DIR is not None:
        candidates.append(Path(MANUAL_DATA_DIR))

    candidates.extend([
        Path("/kaggle/input"),
        Path("/kaggle/input/data-20260607"),
        Path("/kaggle/input/projekt-2"),
        Path("/kaggle/input/projekt2"),
        Path("Data-20260607"),
        Path("Projekt 2/Data-20260607"),
    ])

    for candidate in candidates:
        if not candidate.exists():
            continue

        direct_matches = sorted(candidate.glob("Subject (*.csv"))
        if direct_matches:
            return candidate

        recursive_matches = sorted(candidate.rglob("Subject (*.csv"))
        if recursive_matches:
            return recursive_matches[0].parent

    raise FileNotFoundError("Could not find Subject CSV files. Set MANUAL_DATA_DIR manually.")


def load_dataset(data_dir):
    """Load all subject CSV files while preserving the original trial order inside each file."""
    frames = []
    files = sorted(data_dir.glob("Subject (*.csv"), key=parse_subject_id)

    for file_path in files:
        subject_id = parse_subject_id(file_path)
        frame = pd.read_csv(file_path)
        frame.columns = [column.strip() for column in frame.columns]
        frame["Original_Row_Order"] = np.arange(len(frame))
        frame[SUBJECT_COLUMN] = subject_id
        frame["Weight_kg"] = SUBJECT_WEIGHTS_KG[subject_id]
        frame["Target_kcal_min"] = frame[TARGET_COLUMN] * frame["Weight_kg"] * WKG_TO_KCAL_PER_MIN
        frames.append(frame)

    data = pd.concat(frames, ignore_index=True)
    data = data.sort_values([SUBJECT_COLUMN, "Original_Row_Order"]).reset_index(drop=True)
    return data


def validate_columns(data):
    """Check that all required columns exist."""
    required = [TIME_COLUMN, SUBJECT_COLUMN, ACTIVITY_COLUMN, TARGET_COLUMN] + INPUT_SIGNALS
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def add_segment_ids(data):
    data = data.sort_values([SUBJECT_COLUMN, "Original_Row_Order"]).reset_index(drop=True).copy()

    previous_subject = data[SUBJECT_COLUMN].shift(1)
    previous_activity = data[ACTIVITY_COLUMN].shift(1)
    previous_time = data[TIME_COLUMN].shift(1)

    subject_changed = data[SUBJECT_COLUMN] != previous_subject
    activity_changed = data[ACTIVITY_COLUMN] != previous_activity

    time_difference = data[TIME_COLUMN] - previous_time
    time_not_continuous = (time_difference < 0) | (time_difference > 1.5)
    time_not_continuous = time_not_continuous.fillna(True)

    new_segment = subject_changed | activity_changed | time_not_continuous
    data["Segment ID"] = new_segment.cumsum().astype(int)
    return data


def build_segment_summary(data):
    """Build one summary row per segment."""
    summary = (
        data.groupby("Segment ID")
        .agg(
            Subject=(SUBJECT_COLUMN, "first"),
            Activity_Code=(ACTIVITY_COLUMN, "first"),
            Number_of_Rows=(TIME_COLUMN, "size"),
            Number_of_Targets=(TARGET_COLUMN, "nunique"),
            Start_Time=(TIME_COLUMN, "min"),
            End_Time=(TIME_COLUMN, "max"),
            Target_W_kg=(TARGET_COLUMN, "first"),
            Weight_kg=("Weight_kg", "first"),
            Target_kcal_min=("Target_kcal_min", "first"),
        )
        .reset_index()
    )

    missing_counts = data.groupby("Segment ID")[INPUT_SIGNALS].apply(lambda frame: int(frame.isna().sum().sum()))
    summary["Missing_Input_Values"] = summary["Segment ID"].map(missing_counts)
    return summary


def create_windows(data, input_signals, window_size=30, step_size=10):
    """Create overlapping windows inside each segment."""
    windows = []
    rows = []
    window_id = 0

    for segment_id, segment in data.groupby("Segment ID", sort=True):
        segment = segment.sort_values("Original_Row_Order").reset_index(drop=True)
        if len(segment) < window_size:
            continue

        for start_index in range(0, len(segment) - window_size + 1, step_size):
            end_index = start_index + window_size
            window = segment.loc[start_index:end_index - 1, input_signals].to_numpy(dtype=np.float32)
            windows.append(window)
            rows.append({
                "Window ID": window_id,
                "Segment ID": int(segment_id),
                "Subject": int(segment[SUBJECT_COLUMN].iloc[0]),
                "Activity Code": int(segment[ACTIVITY_COLUMN].iloc[0]),
                "Start Index": int(start_index),
                "End Index": int(end_index),
                "Target_W_kg": float(segment[TARGET_COLUMN].iloc[0]),
                "Weight_kg": float(segment["Weight_kg"].iloc[0]),
                "Target_kcal_min": float(segment["Target_kcal_min"].iloc[0]),
            })
            window_id += 1

    X = np.stack(windows).astype(np.float32)
    metadata = pd.DataFrame(rows)
    y_wkg = metadata["Target_W_kg"].to_numpy(dtype=np.float32)
    return X, y_wkg, metadata


DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR)

raw_data = load_dataset(DATA_DIR)
validate_columns(raw_data)
data = add_segment_ids(raw_data)
segment_summary = build_segment_summary(data)

if not (segment_summary["Number_of_Targets"] == 1).all():
    raise ValueError("At least one segment has more than one target value.")

X_raw, y_wkg, metadata = create_windows(data, INPUT_SIGNALS, WINDOW_SIZE, STEP_SIZE)

print("Dataset shape:", data.shape)
print("Number of segments:", data["Segment ID"].nunique())
print("Window shape:", X_raw.shape)
display(segment_summary["Number_of_Rows"].describe())
display(metadata.head())

# Hard sanity checks from Notebook 01/02. Stop immediately if segmentation is wrong.
assert data["Segment ID"].nunique() == 198, "Expected 198 continuous segments/trials."
assert X_raw.shape[0] == 6724, "Expected 6,724 windows with window size 30 and step size 10."


## 4. LOSO Splits and Normalization

In [ ]:
def create_loso_splits(subjects):
    """Create deterministic LOSO splits: one test subject and the next subject as validation."""
    subjects = list(sorted(subjects))
    splits = []
    for fold_index, test_subject in enumerate(subjects):
        validation_subject = subjects[(fold_index + 1) % len(subjects)]
        train_subjects = [subject for subject in subjects if subject not in [test_subject, validation_subject]]
        splits.append({
            "Fold": fold_index + 1,
            "Train Subjects": train_subjects,
            "Validation Subject": validation_subject,
            "Test Subject": test_subject,
        })
    return splits


def get_split_indices(metadata, split):
    """Return train, validation, and test indices for one split."""
    train_mask = metadata["Subject"].isin(split["Train Subjects"]).to_numpy()
    validation_mask = (metadata["Subject"] == split["Validation Subject"]).to_numpy()
    test_mask = (metadata["Subject"] == split["Test Subject"]).to_numpy()
    return np.where(train_mask)[0], np.where(validation_mask)[0], np.where(test_mask)[0]


def normalize_by_training_data(X_train, X_validation, X_test):
    """Standardize channels using training data only."""
    mean = np.nanmean(X_train, axis=(0, 1), keepdims=True)
    std = np.nanstd(X_train, axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)

    X_train = (X_train - mean) / std
    X_validation = (X_validation - mean) / std
    X_test = (X_test - mean) / std

    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    X_validation = np.nan_to_num(X_validation, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return X_train, X_validation, X_test


subjects = sorted(metadata["Subject"].unique())
loso_splits = create_loso_splits(subjects)
display(pd.DataFrame(loso_splits))

for split in loso_splits:
    train_idx, validation_idx, test_idx = get_split_indices(metadata, split)
    assert set(metadata.iloc[train_idx]["Subject"]).isdisjoint(set(metadata.iloc[validation_idx]["Subject"]))
    assert set(metadata.iloc[train_idx]["Subject"]).isdisjoint(set(metadata.iloc[test_idx]["Subject"]))
    assert set(metadata.iloc[validation_idx]["Subject"]).isdisjoint(set(metadata.iloc[test_idx]["Subject"]))

print("All LOSO splits passed leakage validation.")

## 5. Metrics

In [ ]:
def rmse(y_true, y_pred):
    """Compute root mean squared error."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred, prefix):
    """Compute regression metrics with a prefix."""
    return {
        f"{prefix}_RMSE_W_kg": rmse(y_true, y_pred),
        f"{prefix}_MAE_W_kg": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}_R2_W_kg": float(r2_score(y_true, y_pred)),
    }


def aggregate_to_segments(prediction_frame):
    """Average window predictions per segment."""
    segment_predictions = (
        prediction_frame.groupby(["Segment ID", "Subject", "Activity Code"], as_index=False)
        .agg(
            Actual_W_kg=("Actual_W_kg", "first"),
            Predicted_W_kg=("Predicted_W_kg", "mean"),
            Weight_kg=("Weight_kg", "first"),
        )
    )
    segment_predictions["Actual_kcal_min"] = segment_predictions["Actual_W_kg"] * segment_predictions["Weight_kg"] * WKG_TO_KCAL_PER_MIN
    segment_predictions["Predicted_kcal_min"] = segment_predictions["Predicted_W_kg"] * segment_predictions["Weight_kg"] * WKG_TO_KCAL_PER_MIN
    return segment_predictions


def evaluate_predictions(metadata_subset, y_true, y_pred, prefix):
    """Evaluate predictions on window and segment level."""
    frame = metadata_subset.copy().reset_index(drop=True)
    frame["Actual_W_kg"] = y_true
    frame["Predicted_W_kg"] = y_pred
    segments = aggregate_to_segments(frame)

    metrics = {}
    metrics.update(regression_metrics(y_true, y_pred, f"{prefix}_Window"))
    metrics.update(regression_metrics(segments["Actual_W_kg"], segments["Predicted_W_kg"], f"{prefix}_Segment"))
    return metrics, frame, segments

## 6. Five Models

In [ ]:
def compile_regression_model(model, learning_rate=1e-3):
    """Compile a Keras regression model."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae"), keras.metrics.RootMeanSquaredError(name="rmse")],
        jit_compile=False,
    )
    return model


def build_simple_fcn(input_shape, filters=32, kernel_size=5, learning_rate=1e-3):
    """Build a simple FCN-style 1D CNN baseline."""
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(1)(x)
    return compile_regression_model(keras.Model(inputs, outputs, name="simple_fcn"), learning_rate)


def residual_cnn_block(x, filters, kernel_size, dropout):
    """Build one residual CNN block."""
    shortcut = x
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)
    x = layers.Add()([shortcut, x])
    return layers.Activation("relu")(x)


def build_residual_cnn(input_shape, filters=32, kernel_size=5, dropout=0.10, learning_rate=1e-3):
    """Build a compact residual CNN."""
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(inputs)
    x = residual_cnn_block(x, filters, kernel_size, dropout)
    x = residual_cnn_block(x, filters, kernel_size, dropout)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1)(x)
    return compile_regression_model(keras.Model(inputs, outputs, name="residual_cnn"), learning_rate)


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout):
    """Build one residual TCN block with dilated convolutions."""
    shortcut = x
    x = layers.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)
    x = layers.Add()([shortcut, x])
    return layers.Activation("relu")(x)


def build_tiny_residual_tcn(input_shape, filters=32, kernel_size=3, dropout=0.10, learning_rate=1e-3):
    """Build a compact residual TCN."""
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for dilation_rate in [1, 2, 4]:
        x = tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1)(x)
    return compile_regression_model(keras.Model(inputs, outputs, name="tiny_residual_tcn"), learning_rate)


def inception_lite_block(x, filters, dropout):
    """Build one lightweight Inception-style temporal block."""
    branch_3 = layers.Conv1D(filters, 3, padding="same", activation="relu")(x)
    branch_5 = layers.Conv1D(filters, 5, padding="same", activation="relu")(x)
    branch_9 = layers.Conv1D(filters, 9, padding="same", activation="relu")(x)
    pooled = layers.MaxPooling1D(pool_size=3, strides=1, padding="same")(x)
    branch_pool = layers.Conv1D(filters, 1, padding="same", activation="relu")(pooled)
    x = layers.Concatenate()([branch_3, branch_5, branch_9, branch_pool])
    x = layers.BatchNormalization()(x)
    return layers.Dropout(dropout)(x)


def build_inception_lite(input_shape, filters=16, dropout=0.10, learning_rate=1e-3):
    """Build a small InceptionTime-inspired model."""
    inputs = keras.Input(shape=input_shape)
    x = inception_lite_block(inputs, filters, dropout)
    x = inception_lite_block(x, filters, dropout)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1)(x)
    return compile_regression_model(keras.Model(inputs, outputs, name="inception_lite"), learning_rate)


def build_small_gru(input_shape, units=32, dropout=0.10, learning_rate=1e-3):
    """Build a small recurrent comparison model."""
    inputs = keras.Input(shape=input_shape)
    x = layers.GRU(units, dropout=dropout, recurrent_dropout=0.0)(inputs)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1)(x)
    return compile_regression_model(keras.Model(inputs, outputs, name="small_gru"), learning_rate)


def build_model_from_config(config, input_shape):
    """Build a model from a configuration dictionary."""
    name = config["name"]
    params = config["params"]
    if name == "simple_fcn":
        return build_simple_fcn(input_shape, **params)
    if name == "residual_cnn":
        return build_residual_cnn(input_shape, **params)
    if name == "tiny_residual_tcn":
        return build_tiny_residual_tcn(input_shape, **params)
    if name == "inception_lite":
        return build_inception_lite(input_shape, **params)
    if name == "small_gru":
        return build_small_gru(input_shape, **params)
    raise ValueError(f"Unknown model: {name}")


MODEL_CONFIGS = [
    {"name": "simple_fcn", "reason": "FCN baseline for raw time-series windows.", "params": {"filters": 32, "kernel_size": 5, "learning_rate": 1e-3}},
    {"name": "residual_cnn", "reason": "Residual CNN tests whether skip connections improve stable feature learning.", "params": {"filters": 32, "kernel_size": 5, "dropout": 0.10, "learning_rate": 1e-3}},
    {"name": "tiny_residual_tcn", "reason": "TCN uses residual blocks and dilated convolutions for temporal context.", "params": {"filters": 32, "kernel_size": 3, "dropout": 0.10, "learning_rate": 1e-3}},
    {"name": "inception_lite", "reason": "Multi-scale kernels capture temporal patterns at different durations.", "params": {"filters": 16, "dropout": 0.10, "learning_rate": 1e-3}},
    {"name": "small_gru", "reason": "Small recurrent comparison model.", "params": {"units": 32, "dropout": 0.10, "learning_rate": 1e-3}},
]

display(pd.DataFrame([{"Model": c["name"], "Reason": c["reason"], "Params": json.dumps(c["params"])} for c in MODEL_CONFIGS]))

## 7. Training and Evaluation Helpers

In [ ]:
def predict_in_batches(model, X, batch_size=512):
    """Predict via direct model calls to avoid repeated predict graph compilation."""
    predictions = []
    for start in range(0, len(X), batch_size):
        batch = tf.convert_to_tensor(X[start:start + batch_size], dtype=tf.float32)
        predictions.append(model(batch, training=False).numpy().reshape(-1))
    return np.concatenate(predictions)


def train_one_fold(config, split, max_epochs=100, patience=12, batch_size=64):
    """Train and evaluate one model on one LOSO fold."""
    tf.keras.backend.clear_session()
    gc.collect()

    train_idx, validation_idx, test_idx = get_split_indices(metadata, split)
    X_train, X_validation, X_test = normalize_by_training_data(
        X_raw[train_idx], X_raw[validation_idx], X_raw[test_idx]
    )
    y_train = y_wkg[train_idx]
    y_validation = y_wkg[validation_idx]
    y_test = y_wkg[test_idx]

    model = build_model_from_config(config, input_shape=X_raw.shape[1:])
    parameter_count = int(model.count_params())

    callbacks = [keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
        min_delta=1e-4,
    )]

    start_time = time.time()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=max_epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0,
    )
    training_time = time.time() - start_time

    validation_pred = predict_in_batches(model, X_validation)
    test_pred = predict_in_batches(model, X_test)

    validation_metrics, validation_windows, validation_segments = evaluate_predictions(
        metadata.iloc[validation_idx], y_validation, validation_pred, "Validation"
    )
    test_metrics, test_windows, test_segments = evaluate_predictions(
        metadata.iloc[test_idx], y_test, test_pred, "Test"
    )

    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    result = {
        "Model": config["name"],
        "Fold": split["Fold"],
        "Validation Subject": split["Validation Subject"],
        "Test Subject": split["Test Subject"],
        "Best Epoch": best_epoch,
        "Trained Epochs": len(history.history["loss"]),
        "Training Time Seconds": training_time,
        "Number of Parameters": parameter_count,
    }
    result.update(validation_metrics)
    result.update(test_metrics)

    history_frame = pd.DataFrame(history.history)
    history_frame["Model"] = config["name"]
    history_frame["Fold"] = split["Fold"]
    history_frame["Epoch"] = np.arange(1, len(history_frame) + 1)

    validation_segments["Model"] = config["name"]
    validation_segments["Fold"] = split["Fold"]
    test_segments["Model"] = config["name"]
    test_segments["Fold"] = split["Fold"]

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return result, history_frame, validation_segments, test_segments

## 8. HPO Design

In [ ]:
# Install/import Optuna.
try:
    import optuna
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna

from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Optuna version:", optuna.__version__)

## 9. Tuning Configuration

In [ ]:
# Hyperparameter optimization budget
COARSE_TRIALS_PER_MODEL = 12
REFINED_TRIALS_PER_MODEL = 8

COARSE_FOLD_NUMBERS = [1, 5, 9]
REFINED_FOLD_NUMBERS = [1, 3, 5, 7, 9]
FINAL_FOLD_NUMBERS = list(range(1, 11))

MAX_EPOCHS_COARSE = 80
PATIENCE_COARSE = 8

MAX_EPOCHS_REFINED = 100
PATIENCE_REFINED = 10

MAX_EPOCHS_FINAL = 100
PATIENCE_FINAL = 12

STORAGE_URL = f"sqlite:///{OUTPUT_DIR / 'optuna_03c_tuning.db'}"

print("Coarse folds:", COARSE_FOLD_NUMBERS)
print("Refined folds:", REFINED_FOLD_NUMBERS)
print("Final folds:", FINAL_FOLD_NUMBERS)
print("Optuna storage:", STORAGE_URL)

## 10. Tunable Model Builders

In [ ]:
def compile_tuned_regression_model(model, learning_rate):
    """Compile a regression model with Adam and MSE loss."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae"), keras.metrics.RootMeanSquaredError(name="rmse")],
        jit_compile=False,
    )
    return model


def build_small_gru_tuned(input_shape, units, num_gru_layers, dense_units, dropout, l2_strength, learning_rate):
    """Build a tunable GRU regression model."""
    regularizer = keras.regularizers.l2(l2_strength) if l2_strength > 0 else None

    inputs = keras.Input(shape=input_shape)
    x = inputs

    for layer_index in range(num_gru_layers):
        return_sequences = layer_index < num_gru_layers - 1
        x = layers.GRU(
            units,
            return_sequences=return_sequences,
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=regularizer,
        )(x)

    x = layers.Dense(dense_units, activation="relu", kernel_regularizer=regularizer)(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1)(x)

    model = keras.Model(inputs, outputs, name="small_gru_tuned")
    return compile_tuned_regression_model(model, learning_rate)


def tcn_residual_block_tuned(x, filters, kernel_size, dilation_rate, dropout, regularizer):
    """Build one tunable residual TCN block."""
    shortcut = x

    x = layers.Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu",
        kernel_regularizer=regularizer,
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        kernel_regularizer=regularizer,
    )(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same", kernel_regularizer=regularizer)(shortcut)

    x = layers.Add()([shortcut, x])
    x = layers.Activation("relu")(x)
    return x


def build_tiny_residual_tcn_tuned(input_shape, filters, kernel_size, dilation_pattern, dense_units, dropout, l2_strength, learning_rate):
    """Build a tunable compact residual TCN."""
    regularizer = keras.regularizers.l2(l2_strength) if l2_strength > 0 else None

    inputs = keras.Input(shape=input_shape)
    x = inputs

    for dilation_rate in dilation_pattern:
        x = tcn_residual_block_tuned(x, filters, kernel_size, dilation_rate, dropout, regularizer)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(dense_units, activation="relu", kernel_regularizer=regularizer)(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1)(x)

    model = keras.Model(inputs, outputs, name="tiny_residual_tcn_tuned")
    return compile_tuned_regression_model(model, learning_rate)


def build_tuned_model(config, input_shape):
    """Build a tuned model from a config dictionary."""
    model_type = config["model_type"]
    params = config["params"]

    if model_type == "small_gru":
        return build_small_gru_tuned(input_shape=input_shape, **params)

    if model_type == "tiny_residual_tcn":
        return build_tiny_residual_tcn_tuned(input_shape=input_shape, **params)

    raise ValueError(f"Unknown model_type: {model_type}")

## 11. Search Spaces

In [ ]:
DILATION_PATTERNS = {
    "1_2": [1, 2],
    "1_2_4": [1, 2, 4],
    "1_2_4_8": [1, 2, 4, 8],
}


def suggest_coarse_config(trial, model_type):
    """Suggest a broad but plausible hyperparameter configuration."""
    if model_type == "small_gru":
        params = {
            "units": trial.suggest_categorical("units", [16, 32, 48, 64, 96]),
            "num_gru_layers": trial.suggest_int("num_gru_layers", 1, 2),
            "dense_units": trial.suggest_categorical("dense_units", [16, 32, 64]),
            "dropout": trial.suggest_float("dropout", 0.0, 0.35),
            "l2_strength": trial.suggest_float("l2_strength", 1e-7, 1e-3, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True),
        }
    elif model_type == "tiny_residual_tcn":
        dilation_key = trial.suggest_categorical("dilation_pattern", list(DILATION_PATTERNS.keys()))
        params = {
            "filters": trial.suggest_categorical("filters", [16, 24, 32, 48, 64]),
            "kernel_size": trial.suggest_categorical("kernel_size", [2, 3, 5, 7]),
            "dilation_pattern": DILATION_PATTERNS[dilation_key],
            "dense_units": trial.suggest_categorical("dense_units", [16, 32, 64]),
            "dropout": trial.suggest_float("dropout", 0.0, 0.35),
            "l2_strength": trial.suggest_float("l2_strength", 1e-7, 1e-3, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True),
        }
    else:
        raise ValueError(model_type)

    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    return {"name": f"{model_type}_coarse_trial_{trial.number}", "model_type": model_type, "params": params, "batch_size": batch_size}


def categorical_neighborhood(value, ordered_values):
    """Return a small categorical neighborhood around a best value."""
    if value not in ordered_values:
        return ordered_values
    index = ordered_values.index(value)
    start = max(0, index - 1)
    end = min(len(ordered_values), index + 2)
    return ordered_values[start:end]


def suggest_float_near(trial, name, best_value, lower_bound, upper_bound, width, log=False):
    """Suggest a float in a local neighborhood around a previous best value."""
    low = max(lower_bound, best_value / width if log else best_value - width)
    high = min(upper_bound, best_value * width if log else best_value + width)
    if low >= high:
        low, high = lower_bound, upper_bound
    return trial.suggest_float(name, low, high, log=log)


def suggest_refined_config(trial, model_type, best_params, best_batch_size):
    """Suggest a refined configuration near the best coarse result."""
    if model_type == "small_gru":
        params = {
            "units": trial.suggest_categorical("units", categorical_neighborhood(best_params["units"], [16, 32, 48, 64, 96])),
            "num_gru_layers": trial.suggest_categorical("num_gru_layers", categorical_neighborhood(best_params["num_gru_layers"], [1, 2])),
            "dense_units": trial.suggest_categorical("dense_units", categorical_neighborhood(best_params["dense_units"], [16, 32, 64])),
            "dropout": suggest_float_near(trial, "dropout", best_params["dropout"], 0.0, 0.45, 0.10),
            "l2_strength": suggest_float_near(trial, "l2_strength", best_params["l2_strength"], 1e-8, 3e-3, 3.0, log=True),
            "learning_rate": suggest_float_near(trial, "learning_rate", best_params["learning_rate"], 1e-4, 5e-3, 2.0, log=True),
        }
    elif model_type == "tiny_residual_tcn":
        best_dilation_key = "_".join(str(value) for value in best_params["dilation_pattern"])
        dilation_key = trial.suggest_categorical("dilation_pattern", categorical_neighborhood(best_dilation_key, list(DILATION_PATTERNS.keys())))
        params = {
            "filters": trial.suggest_categorical("filters", categorical_neighborhood(best_params["filters"], [16, 24, 32, 48, 64])),
            "kernel_size": trial.suggest_categorical("kernel_size", categorical_neighborhood(best_params["kernel_size"], [2, 3, 5, 7])),
            "dilation_pattern": DILATION_PATTERNS[dilation_key],
            "dense_units": trial.suggest_categorical("dense_units", categorical_neighborhood(best_params["dense_units"], [16, 32, 64])),
            "dropout": suggest_float_near(trial, "dropout", best_params["dropout"], 0.0, 0.45, 0.10),
            "l2_strength": suggest_float_near(trial, "l2_strength", best_params["l2_strength"], 1e-8, 3e-3, 3.0, log=True),
            "learning_rate": suggest_float_near(trial, "learning_rate", best_params["learning_rate"], 1e-4, 5e-3, 2.0, log=True),
        }
    else:
        raise ValueError(model_type)

    batch_size = trial.suggest_categorical("batch_size", categorical_neighborhood(best_batch_size, [32, 64, 128]))
    return {"name": f"{model_type}_refined_trial_{trial.number}", "model_type": model_type, "params": params, "batch_size": batch_size}

## 12. Tuning Evaluation Helpers

In [ ]:
def get_splits_by_fold_numbers(fold_numbers):
    """Select LOSO split dictionaries by fold number."""
    fold_set = set(fold_numbers)
    return [split for split in loso_splits if split["Fold"] in fold_set]


def train_evaluate_tuned_config_on_split(config, split, max_epochs, patience):
    """Train one tuned config on one split and return metrics, history, and predictions."""
    tf.keras.backend.clear_session()
    gc.collect()

    train_idx, validation_idx, test_idx = get_split_indices(metadata, split)
    X_train, X_validation, X_test = normalize_by_training_data(X_raw[train_idx], X_raw[validation_idx], X_raw[test_idx])
    y_train = y_wkg[train_idx]
    y_validation = y_wkg[validation_idx]
    y_test = y_wkg[test_idx]

    model = build_tuned_model(config, input_shape=X_raw.shape[1:])
    parameter_count = int(model.count_params())

    callbacks = [keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
        min_delta=1e-4,
    )]

    start_time = time.time()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=max_epochs,
        batch_size=config["batch_size"],
        callbacks=callbacks,
        verbose=0,
    )
    training_time = time.time() - start_time

    validation_pred = predict_in_batches(model, X_validation)
    test_pred = predict_in_batches(model, X_test)

    validation_metrics, _, validation_segments = evaluate_predictions(metadata.iloc[validation_idx], y_validation, validation_pred, "Validation")
    test_metrics, _, test_segments = evaluate_predictions(metadata.iloc[test_idx], y_test, test_pred, "Test")

    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    result = {
        "Config": config["name"],
        "Model Type": config["model_type"],
        "Fold": split["Fold"],
        "Validation Subject": split["Validation Subject"],
        "Test Subject": split["Test Subject"],
        "Best Epoch": best_epoch,
        "Trained Epochs": len(history.history["loss"]),
        "Training Time Seconds": training_time,
        "Number of Parameters": parameter_count,
        "Batch Size": config["batch_size"],
    }
    result.update(validation_metrics)
    result.update(test_metrics)

    history_frame = pd.DataFrame(history.history)
    history_frame["Config"] = config["name"]
    history_frame["Model Type"] = config["model_type"]
    history_frame["Fold"] = split["Fold"]
    history_frame["Epoch"] = np.arange(1, len(history_frame) + 1)

    validation_segments["Config"] = config["name"]
    validation_segments["Model Type"] = config["model_type"]
    validation_segments["Fold"] = split["Fold"]

    test_segments["Config"] = config["name"]
    test_segments["Model Type"] = config["model_type"]
    test_segments["Fold"] = split["Fold"]

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return result, history_frame, validation_segments, test_segments


def objective_for_model(model_type, phase, fold_numbers, max_epochs, patience, best_config=None):
    """Create an Optuna objective function for a model and phase."""
    selected_splits = get_splits_by_fold_numbers(fold_numbers)

    def objective(trial):
        if phase == "coarse":
            config = suggest_coarse_config(trial, model_type)
        elif phase == "refined":
            config = suggest_refined_config(
                trial,
                model_type,
                best_params=best_config["params"],
                best_batch_size=best_config["batch_size"],
            )
        else:
            raise ValueError(phase)

        trial.set_user_attr("config_json", json.dumps(config))
        fold_scores = []

        for step, split in enumerate(selected_splits, start=1):
            result, _, _, _ = train_evaluate_tuned_config_on_split(config, split, max_epochs=max_epochs, patience=patience)
            fold_scores.append(result["Validation_Segment_RMSE_W_kg"])
            current_mean = float(np.mean(fold_scores))

            trial.report(current_mean, step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_scores))

    return objective


def create_or_load_study(study_name, n_startup_trials):
    """Create or load an Optuna study with TPE sampler and median pruning."""
    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=n_startup_trials,
        multivariate=True,
        group=True,
    )
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=max(4, n_startup_trials // 2),
        n_warmup_steps=1,
    )
    return optuna.create_study(
        study_name=study_name,
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        storage=STORAGE_URL,
        load_if_exists=True,
    )


def run_study(study_name, objective, n_trials):
    """Run a study until it reaches n_trials completed/pruned/failed trials."""
    study = create_or_load_study(study_name, n_startup_trials=min(6, n_trials))
    existing_trials = len(study.trials)
    remaining_trials = max(0, n_trials - existing_trials)
    print(f"Study {study_name}: existing trials={existing_trials}, remaining={remaining_trials}")

    if remaining_trials > 0:
        study.optimize(objective, n_trials=remaining_trials, n_jobs=1, gc_after_trial=True)

    return study


def study_to_dataframe(study, model_type, phase):
    """Convert an Optuna study to a compact DataFrame."""
    rows = []
    for trial in study.trials:
        row = {
            "Model Type": model_type,
            "Phase": phase,
            "Trial": trial.number,
            "State": str(trial.state),
            "Value": trial.value,
        }
        row.update(trial.params)
        rows.append(row)
    return pd.DataFrame(rows)


def best_config_from_study(study):
    """Read the config JSON stored in the best completed trial."""
    config_json = study.best_trial.user_attrs["config_json"]
    return json.loads(config_json)

## 13. Phase 1: Coarse Search

In [ ]:
coarse_studies = {}
coarse_frames = []

for model_type in ["small_gru", "tiny_residual_tcn"]:
    study_name = f"03c_coarse_{model_type}"
    objective = objective_for_model(
        model_type=model_type,
        phase="coarse",
        fold_numbers=COARSE_FOLD_NUMBERS,
        max_epochs=MAX_EPOCHS_COARSE,
        patience=PATIENCE_COARSE,
    )
    study = run_study(study_name, objective, COARSE_TRIALS_PER_MODEL)
    coarse_studies[model_type] = study
    coarse_frames.append(study_to_dataframe(study, model_type, "coarse"))

coarse_results = pd.concat(coarse_frames, ignore_index=True)
coarse_results.to_csv(OUTPUT_DIR / "coarse_optuna_trials.csv", index=False)

display(coarse_results.sort_values("Value").head(20))

best_coarse_configs = {model_type: best_config_from_study(study) for model_type, study in coarse_studies.items()}
print(json.dumps(best_coarse_configs, indent=2))

## 14. Phase 2: Refined Search

In [ ]:
refined_studies = {}
refined_frames = []

for model_type in ["small_gru", "tiny_residual_tcn"]:
    study_name = f"03c_refined_{model_type}"
    objective = objective_for_model(
        model_type=model_type,
        phase="refined",
        fold_numbers=REFINED_FOLD_NUMBERS,
        max_epochs=MAX_EPOCHS_REFINED,
        patience=PATIENCE_REFINED,
        best_config=best_coarse_configs[model_type],
    )
    study = run_study(study_name, objective, REFINED_TRIALS_PER_MODEL)
    refined_studies[model_type] = study
    refined_frames.append(study_to_dataframe(study, model_type, "refined"))

refined_results = pd.concat(refined_frames, ignore_index=True)
refined_results.to_csv(OUTPUT_DIR / "refined_optuna_trials.csv", index=False)

display(refined_results.sort_values("Value").head(20))

best_refined_configs = {model_type: best_config_from_study(study) for model_type, study in refined_studies.items()}
print(json.dumps(best_refined_configs, indent=2))

## 15. Final 10-Fold Check

In [ ]:
baseline_small_gru_config = {
    "name": "baseline_small_gru",
    "model_type": "small_gru",
    "batch_size": 64,
    "params": {
        "units": 32,
        "num_gru_layers": 1,
        "dense_units": 32,
        "dropout": 0.10,
        "l2_strength": 0.0,
        "learning_rate": 1e-3,
    },
}

baseline_tcn_config = {
    "name": "baseline_tiny_residual_tcn",
    "model_type": "tiny_residual_tcn",
    "batch_size": 64,
    "params": {
        "filters": 32,
        "kernel_size": 3,
        "dilation_pattern": [1, 2, 4],
        "dense_units": 32,
        "dropout": 0.10,
        "l2_strength": 0.0,
        "learning_rate": 1e-3,
    },
}

tuned_small_gru_config = best_refined_configs["small_gru"]
tuned_small_gru_config["name"] = "optuna_small_gru"

tuned_tcn_config = best_refined_configs["tiny_residual_tcn"]
tuned_tcn_config["name"] = "optuna_tiny_residual_tcn"

final_configs = [
    baseline_small_gru_config,
    tuned_small_gru_config,
    baseline_tcn_config,
    tuned_tcn_config,
]

print(json.dumps(final_configs, indent=2))
with open(OUTPUT_DIR / "final_candidate_configs.json", "w") as file:
    json.dump(final_configs, file, indent=2)

In [ ]:
final_results = []
final_histories = []
final_validation_segments = []
final_test_segments = []

for config in final_configs:
    print("=" * 100)
    print("Final check config:", config["name"])

    for split in get_splits_by_fold_numbers(FINAL_FOLD_NUMBERS):
        print(
            f"Training {config['name']} | fold {split['Fold']} | "
            f"validation subject {split['Validation Subject']} | test subject {split['Test Subject']}"
        )

        result, history_frame, validation_segments, test_segments = train_evaluate_tuned_config_on_split(
            config=config,
            split=split,
            max_epochs=MAX_EPOCHS_FINAL,
            patience=PATIENCE_FINAL,
        )

        final_results.append(result)
        final_histories.append(history_frame)
        final_validation_segments.append(validation_segments)
        final_test_segments.append(test_segments)

        pd.DataFrame(final_results).to_csv(OUTPUT_DIR / "final_check_fold_results_incremental.csv", index=False)
        pd.concat(final_histories, ignore_index=True).to_csv(OUTPUT_DIR / "final_check_learning_curves_incremental.csv", index=False)

        print(
            "Validation Segment RMSE:", round(result["Validation_Segment_RMSE_W_kg"], 3),
            "| Test Segment RMSE:", round(result["Test_Segment_RMSE_W_kg"], 3),
            "| Best epoch:", result["Best Epoch"],
        )

final_results = pd.DataFrame(final_results)
final_histories = pd.concat(final_histories, ignore_index=True)
final_validation_segments = pd.concat(final_validation_segments, ignore_index=True)
final_test_segments = pd.concat(final_test_segments, ignore_index=True)

final_results.to_csv(OUTPUT_DIR / "final_check_fold_results.csv", index=False)
final_histories.to_csv(OUTPUT_DIR / "final_check_learning_curves.csv", index=False)
final_validation_segments.to_csv(OUTPUT_DIR / "final_validation_segment_predictions.csv", index=False)
final_test_segments.to_csv(OUTPUT_DIR / "final_test_segment_predictions.csv", index=False)

display(final_results.head())

## 16. Final Ranking and Selection

In [ ]:
final_ranking = (
    final_results.groupby(["Config", "Model Type"])
    .agg(
        Mean_Validation_Segment_RMSE_W_kg=("Validation_Segment_RMSE_W_kg", "mean"),
        Std_Validation_Segment_RMSE_W_kg=("Validation_Segment_RMSE_W_kg", "std"),
        Mean_Validation_Segment_MAE_W_kg=("Validation_Segment_MAE_W_kg", "mean"),
        Mean_Validation_Segment_R2_W_kg=("Validation_Segment_R2_W_kg", "mean"),
        Mean_Test_Segment_RMSE_W_kg=("Test_Segment_RMSE_W_kg", "mean"),
        Std_Test_Segment_RMSE_W_kg=("Test_Segment_RMSE_W_kg", "std"),
        Mean_Test_Segment_R2_W_kg=("Test_Segment_R2_W_kg", "mean"),
        Mean_Best_Epoch=("Best Epoch", "mean"),
        Mean_Trained_Epochs=("Trained Epochs", "mean"),
        Mean_Training_Time_Seconds=("Training Time Seconds", "mean"),
        Mean_Number_of_Parameters=("Number of Parameters", "mean"),
    )
    .reset_index()
    .sort_values("Mean_Validation_Segment_RMSE_W_kg")
)

final_ranking.to_csv(OUTPUT_DIR / "final_optuna_ranking.csv", index=False)
display(final_ranking)

# Selection rule: validation RMSE first; if nearly tied, prefer lower standard deviation and smaller model.
SELECTION_TOLERANCE_W_KG = 0.02
best_rmse = final_ranking["Mean_Validation_Segment_RMSE_W_kg"].min()
selection_pool = final_ranking[
    final_ranking["Mean_Validation_Segment_RMSE_W_kg"] <= best_rmse + SELECTION_TOLERANCE_W_KG
].copy()
selection_pool = selection_pool.sort_values([
    "Std_Validation_Segment_RMSE_W_kg",
    "Mean_Number_of_Parameters",
    "Mean_Validation_Segment_RMSE_W_kg",
])

selected_config_name = selection_pool.iloc[0]["Config"]
selected_config = next(config for config in final_configs if config["name"] == selected_config_name)

selection_summary = {
    "selected_config_name": selected_config_name,
    "selection_tolerance_W_kg": SELECTION_TOLERANCE_W_KG,
    "selected_ranking_row": selection_pool.iloc[0].to_dict(),
    "selected_config": selected_config,
}

with open(OUTPUT_DIR / "selected_optuna_config.json", "w") as file:
    json.dump(selection_summary, file, indent=2)

print("Selected config:", selected_config_name)
print(json.dumps(selection_summary, indent=2))

## 17. Plots

In [ ]:
plt.figure(figsize=(11, 5))
sns.boxplot(data=final_results, x="Config", y="Validation_Segment_RMSE_W_kg", order=final_ranking["Config"])
sns.stripplot(data=final_results, x="Config", y="Validation_Segment_RMSE_W_kg", order=final_ranking["Config"], color="black", alpha=0.65, size=5)
plt.title("Final Check: Validation Segment RMSE Across LOSO Folds")
plt.xlabel("Config")
plt.ylabel("Validation Segment RMSE (W/kg)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "final_validation_rmse_boxplot.png", dpi=160)
plt.show()

plt.figure(figsize=(11, 5))
sns.barplot(data=final_ranking, x="Config", y="Mean_Validation_Segment_RMSE_W_kg", order=final_ranking["Config"])
plt.title("Final Check: Mean Validation Segment RMSE")
plt.xlabel("Config")
plt.ylabel("Mean Validation Segment RMSE (W/kg)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "final_validation_rmse_barplot.png", dpi=160)
plt.show()

In [ ]:
def plot_final_learning_curves(config_name, folds_to_plot=[1, 5, 10]):
    """Plot learning curves for selected folds of a final config."""
    config_history = final_histories[final_histories["Config"] == config_name]

    plt.figure(figsize=(11, 5))
    for fold in folds_to_plot:
        fold_history = config_history[config_history["Fold"] == fold]
        if fold_history.empty:
            continue
        plt.plot(fold_history["Epoch"], fold_history["loss"], linestyle="--", alpha=0.60, label=f"Fold {fold} train")
        plt.plot(fold_history["Epoch"], fold_history["val_loss"], linewidth=2, alpha=0.85, label=f"Fold {fold} val")

    plt.title(f"Learning Curves: {config_name}")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"learning_curves_{config_name}.png", dpi=160)
    plt.show()


for config_name in final_ranking["Config"]:
    plot_final_learning_curves(config_name)

In [ ]:
selected_test_segments = final_test_segments[final_test_segments["Config"] == selected_config_name].copy()

plt.figure(figsize=(7, 7))
sns.scatterplot(
    data=selected_test_segments,
    x="Actual_W_kg",
    y="Predicted_W_kg",
    hue="Subject",
    palette="tab10",
    s=55,
)
min_value = min(selected_test_segments["Actual_W_kg"].min(), selected_test_segments["Predicted_W_kg"].min())
max_value = max(selected_test_segments["Actual_W_kg"].max(), selected_test_segments["Predicted_W_kg"].max())
plt.plot([min_value, max_value], [min_value, max_value], "k--", label="Perfect Prediction")
plt.title(f"Predicted vs Actual EE: {selected_config_name}")
plt.xlabel("Actual EE (W/kg)")
plt.ylabel("Predicted EE (W/kg)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"predicted_vs_actual_{selected_config_name}.png", dpi=160)
plt.show()